In [1]:
#Imports 
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from ID3 import ID3Classifier
from sklearn.metrics import f1_score

ruta = "futbol_uruguayo.csv"
ds = pd.read_csv(ruta)

In [2]:
#Preprocesamiento
#-1 Eligo los atributos del dataset
atributos = ["home_ident", "away_ident", "historial", "nivel_relativo"]

#0. Genero un split
tscv = TimeSeriesSplit(n_splits=3) #Con kfoldin habria data leakage 

#1. Calculo columna resultado
ds["resultado"] = np.select(
    [ds["gh"] > ds["ga"], ds["gh"] < ds["ga"]], ["G", "P"], default="E"
)
ds["date"] = pd.to_datetime(ds["date"])

#2. Creacion de nuevas columnas de datos: historial, nivelRelativo 
def calcular_historial_y_nivel(ds):
    # Ordenamos cronológicamente
    ds = ds.sort_values("date").copy()

    # Nuevas columnas
    ds["historial"] = "Neutro"
    ds["nivel_relativo"] = "Parejo"

    # Recorremos cada partido
    for i, fila in ds.iterrows():

        local = fila["home_ident"]
        visitante = fila["away_ident"]
        fecha = fila["date"]

        # -----------------------------------------
        # Buscar enfrentamientos anteriores
        # entre estos dos equipos
        # -----------------------------------------

        anteriores = ds.loc[
            (ds["date"] < fecha) &
            (
                ((ds["home_ident"] == local) & (ds["away_ident"] == visitante)) |
                ((ds["home_ident"] == visitante) & (ds["away_ident"] == local))
            )
        ].tail(10)

        # Si no hay enfrentamientos anteriores
        if len(anteriores) == 0:
            continue

        victorias = 0
        derrotas = 0
        goles_favor = []
        goles_contra = []

        # -----------------------------------------
        # Analizamos los últimos enfrentamientos
        # desde la perspectiva del equipo LOCAL
        # de la fila actual
        # -----------------------------------------

        for _, partido in anteriores.iterrows():

            if (
                partido["home_ident"] == local
                and partido["away_ident"] == visitante
            ):
                gf = partido["gh"]
                gc = partido["ga"]
            else:
                # El equipo actual jugó de visitante
                gf = partido["ga"]
                gc = partido["gh"]

            goles_favor.append(gf)
            goles_contra.append(gc)

            if gf > gc:
                victorias += 1
            elif gf < gc:
                derrotas += 1

        # -----------------------------------------
        # 1. Historial
        # -----------------------------------------

        if victorias > derrotas:
            ds.at[i, "historial"] = "Positivo"
        elif derrotas > victorias:
            ds.at[i, "historial"] = "Negativo"
        else:
            ds.at[i, "historial"] = "Neutro"

        # -----------------------------------------
        # 2. Nivel relativo (diferencia de forma reciente)
        # -----------------------------------------

        promedio_gf = sum(goles_favor) / len(goles_favor)
        promedio_gc = sum(goles_contra) / len(goles_contra)
        diferencia = promedio_gf - promedio_gc

        if diferencia <= -1.5:
            categoria_nivel = "Muy_Inferior"
        elif diferencia <= -0.5:
            categoria_nivel = "Inferior"
        elif diferencia < 0.5:
            categoria_nivel = "Parejo"
        elif diferencia < 1.5:
            categoria_nivel = "Superior"
        else:
            categoria_nivel = "Muy_Superior"

        ds.at[i, "nivel_relativo"] = categoria_nivel

    return ds
ds = calcular_historial_y_nivel(ds)

#3. Separacion en datos de Entrenamiento y Test
mask_test = ds["date"].dt.year >= 2024
train = ds[~mask_test]
test = ds[mask_test]

X_train = train[atributos] 
y_train = train["resultado"]

X_test = test[atributos]
y_test = test["resultado"]


#4. Aplico pipeline para el resto del preprocessing
atributes_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer([
    ('atributes', atributes_pipeline, atributos)
])

In [15]:
#Estimador 1 (sklearn - Random Forest)
pipeline_rf = Pipeline([
    ('preprocessing', preprocessing),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    "model__n_estimators": [10, 25, 50, 75, 100, 150],
    "model__max_depth": [1, 3, 5, 7, None],
}

random_rf = RandomizedSearchCV(
    estimator= pipeline_rf,
    param_distributions=param_grid_rf,
    cv=tscv,
    scoring="f1_macro",
    n_jobs=-1,
    n_iter=24,
    random_state=42
)

random_rf.fit(X_train, y_train)

y_pred_rf = random_rf.predict(X_test)

In [13]:
#Estimador 2 (sklearn - Naive Bayes)
pipeline_nb = Pipeline([
    ('preprocessing', preprocessing),
    ('model', CategoricalNB())
])

param_grid_nb = {"model__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]} #Suavizado de Laplace (dependiendo del alpha creo mas o menos instancias de cada partido)

random_nb = RandomizedSearchCV(
    estimator=pipeline_nb, 
    param_distributions=param_grid_nb, 
    cv=tscv, 
    scoring="accuracy",
    n_jobs=-1,
    n_iter=5,
    random_state=42
)
random_nb.fit(X_train, y_train)

y_pred_nb = random_nb.predict(X_test)

In [3]:
#Estimador 3 (nuestro - ID3)

anios_validacion = [2019, 2020, 2021, 2022, 2023]

valores_min_info_gain = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.1,
]

resultados_id3 = []

for min_info_gain in valores_min_info_gain:

    for anio_validacion in anios_validacion:
        mascara_entrenamiento = (
            train["date"].dt.year < anio_validacion
        )
        mascara_validacion = (
            train["date"].dt.year == anio_validacion
        )

        X_entrenamiento = train.loc[
            mascara_entrenamiento,
            atributos
        ]
        y_entrenamiento = train.loc[
            mascara_entrenamiento,
            "resultado"
        ]

        X_validacion = train.loc[
            mascara_validacion,
            atributos
        ]
        y_validacion = train.loc[
            mascara_validacion,
            "resultado"
        ]

        modelo_fold = ID3Classifier(
            criterio="Ganancia",
            min_info_gain=min_info_gain
        )

        modelo_fold.fit(
            X_entrenamiento,
            y_entrenamiento
        )

        predicciones = modelo_fold.predict(
            X_validacion
        )

        resultados_id3.append({
            "min_info_gain": min_info_gain,
            "anio_validacion": anio_validacion,
            "macro_f1": f1_score(
                y_validacion,
                predicciones,
                average="macro",
                zero_division=0
            ),
            "accuracy": accuracy_score(
                y_validacion,
                predicciones
            )
        })

In [4]:
resultados_id3_df = pd.DataFrame(resultados_id3)

resumen_id3 = (
    resultados_id3_df
    .groupby("min_info_gain", as_index=False)
    .agg(
        macro_f1_promedio=("macro_f1", "mean"),
        macro_f1_desviacion=("macro_f1", "std"),
        accuracy_promedio=("accuracy", "mean")
    )
    .sort_values(
        "macro_f1_promedio",
        ascending=False
    )
)

resumen_id3

,min_info_gain,macro_f1_promedio,macro_f1_desviacion,accuracy_promedio
0,0.001,0.349164,0.018393,0.370351
1,0.005,0.349164,0.018393,0.370351
2,0.010,0.349164,0.018393,0.370351
3,0.020,0.349164,0.018393,0.370351
4,0.050,0.313674,0.024226,0.390331
5,0.100,0.186606,0.010068,0.389207


In [ ]:
#Estimador 4 (nuestro - Naive Bayes)

In [8]:
#Estimador 5 (Resultado mas probable (10 anos))
subset = (ds["date"].dt.year >= 2014) & (ds["date"].dt.year < 2024)
subset_l = ds.loc[subset, "resultado"]

mas_sale = subset.mode()[0]

In [16]:
#Estadisticas
print("Mejores hiperparámetros (Random Forest):", random_rf.best_params_)
print("Accuracy (Random Forest):", accuracy_score(y_test, y_pred_rf))
print("\nReporte de clasificación: (Random Forest)\n", classification_report(y_test, y_pred_rf))

print("Mejores hiperparámetros (Naive Bayes):", random_nb.best_params_)
print("Accuracy (Naive Bayes):", accuracy_score(y_test, y_pred_nb))
print("\nReporte de clasificación: (Naive Bayes)\n", classification_report(y_test, y_pred_nb))

total = (y_test == "G").sum()/len(y_test)
print("Estimador base:", total)

Mejores hiperparámetros (Random Forest): {'model__n_estimators': 10, 'model__max_depth': None}
Accuracy (Random Forest): 0.4439746300211416

Reporte de clasificación: (Random Forest)
               precision    recall  f1-score   support

           E       0.29      0.22      0.25       132
           G       0.51      0.67      0.58       190
           P       0.44      0.36      0.39       151

    accuracy                           0.44       473
   macro avg       0.41      0.42      0.41       473
weighted avg       0.42      0.44      0.43       473

Mejores hiperparámetros (Naive Bayes): {'model__alpha': 2.0}
Accuracy (Naive Bayes): 0.4693446088794926

Reporte de clasificación: (Naive Bayes)
               precision    recall  f1-score   support

           E       0.32      0.08      0.13       132
           G       0.51      0.72      0.60       190
           P       0.43      0.50      0.46       151

    accuracy                           0.47       473
   macro avg     